In [4]:
import os
import logging
import time
import glob
import duckdb
import json
import traceback

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Path for the failure/skip report
REPORT_FILE = r"D:\EmerG-NeT\processing_report.txt"

def log_to_report(message):
    with open(REPORT_FILE, "a") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} - {message}\n")

def rebuild_abstract_duckdb(inverted_index):
    if not inverted_index:
        return ""
    try:
        data = inverted_index
        if isinstance(inverted_index, str):
            data = json.loads(inverted_index)
        if not data:
            return ""
        max_idx = 0
        for indices in data.values():
            if indices:
                local_max = max(indices)
                if local_max > max_idx:
                    max_idx = local_max
        if max_idx > 30000: 
            return ""
        result = [""] * (max_idx + 1)
        for word, indices in data.items():
            for idx in indices:
                if 0 <= idx < len(result):
                    result[idx] = str(word)
        return " ".join(result).strip()
    except Exception:
        return ""

def run_pipeline(input_path: str, output_dir: str):
    start_time = time.time()
    con = duckdb.connect(':memory:')
    con.create_function("rebuild_abstract", rebuild_abstract_duckdb, return_type='VARCHAR')

    col_to_keep = [
        'id', 'display_name', 'publication_date', 'language', 'type',
        'authorships', 'authors_count', 'topics', 'keywords',
        'locations_count', 'primary_location', 'sustainable_development_goals',
        'awards', 'funders', 'countries_distinct_count', 'institutions_distinct_count',
        'open_access', 'is_paratext', 'is_retracted',
        'referenced_works', 'referenced_works_count', 'related_works',
        'abstract_inverted_index', 'cited_by_count', 'counts_by_year', 'fwci',
        'citation_normalized_percentile', 'cited_by_percentile_year', 'mesh'
    ]
    
    cols_str = ", ".join([f'"{c}"' for c in col_to_keep if c != 'abstract_inverted_index'])

    try:
        # --- ENGLISH LANGUAGE CHECK ---
        # We check if there are ANY English records in this specific parquet file
        has_english = con.execute(f"SELECT count(*) FROM read_parquet('{input_path}') WHERE language = 'en'").fetchone()[0]
        
        if has_english == 0:
            msg = f"SKIP: File {input_path} does not contain English language records. Processing aborted."
            logger.warning(msg)
            log_to_report(msg)
            return

        con.execute(f"""
            CREATE VIEW raw_df AS 
            SELECT {cols_str}, abstract_inverted_index::JSON AS abstract_inverted_index
            FROM read_parquet('{input_path}')
        """)
        
        actual_date = None
        parts = os.path.normpath(input_path).split(os.sep)
        date_part = [p for p in parts if "updated_date=" in p]
        if date_part:
            actual_date = date_part[0].split("=")[1]
        date_sql = f"CAST('{actual_date}' AS DATE)" if actual_date else "NULL"

        con.execute(f"""
            CREATE TABLE clean_df AS
            SELECT 
                REPLACE(id, 'https://openalex.org/', '') AS paper_id,
                display_name AS paper_title,
                publication_date,
                type,
                authorships,
                authors_count::BIGINT AS authors_count,
                topics,
                keywords,
                locations_count::BIGINT AS locations_count,
                primary_location,
                sustainable_development_goals AS SDGs,
                awards,
                funders,
                countries_distinct_count::BIGINT AS countries_distinct_count,
                institutions_distinct_count::BIGINT AS institutions_distinct_count,
                open_access,
                referenced_works,
                referenced_works_count::BIGINT AS referenced_works_count,
                related_works,
                abstract_inverted_index,
                cited_by_count::BIGINT AS cited_by_count,
                counts_by_year,
                COALESCE(fwci, 1.0)::FLOAT AS fwci_val,
                citation_normalized_percentile,
                cited_by_percentile_year,
                mesh,
                {date_sql} AS updated_date
            FROM raw_df
            WHERE language = 'en' AND is_paratext = FALSE AND is_retracted = FALSE
        """)

        con.execute("""
            CREATE TABLE processed_df AS
            SELECT 
                * EXCLUDE (open_access, citation_normalized_percentile, cited_by_percentile_year),
                CAST(open_access.is_oa AS BOOLEAN) AS is_oa,
                open_access.oa_status AS oa_status,
                CASE 
                    WHEN cited_by_count = 0 THEN 0.0
                    ELSE COALESCE(CAST(citation_normalized_percentile.value AS FLOAT), 0.0)
                END AS percentile_rank,
                ((COALESCE(cited_by_percentile_year.min::FLOAT, 0.0) + COALESCE(cited_by_percentile_year.max::FLOAT, 0.0)) / 2.0)::FLOAT AS cited_by_percentile_year_avg
            FROM clean_df
        """)

        os.makedirs(output_dir, exist_ok=True)
        
        # --- SAFETY MEASURES FOR PARQUET CREATION ---
        
        queries = {
            "authors_data.parquet": """
                SELECT paper_id, REPLACE(a.author.id, 'https://openalex.org/', '') as author_id 
                FROM (SELECT paper_id, unnest(authorships) as a FROM processed_df)
            """,
            "topic_data.parquet": """
                SELECT paper_id, REPLACE(t.id, 'https://openalex.org/', '') as topic_id 
                FROM (SELECT paper_id, unnest(topics) as t FROM processed_df)
            """,
            "mesh_data.parquet": "SELECT paper_id, mesh FROM processed_df WHERE array_length(mesh) > 0",
            "awards_data.parquet": "SELECT paper_title, SDGs, awards, funders FROM processed_df",
            "prime_location_data.parquet": """
                SELECT paper_id, REPLACE(primary_location.source.id, 'https://openalex.org/', '') AS source_id,
                primary_location.source.display_name AS source_name,
                REPLACE(primary_location.source.host_organization, 'https://openalex.org/', '') AS host_org_id,
                primary_location.source.host_organization_name AS host_org_name
                FROM processed_df WHERE primary_location.source.id IS NOT NULL
            """,
            "cited_data.parquet": """
                SELECT paper_id, REPLACE(unnested_refs, 'https://openalex.org/', '') AS cited_id
                FROM (SELECT paper_id, unnest(referenced_works) AS unnested_refs FROM processed_df)
                WHERE unnested_refs IS NOT NULL
            """,
            "related_data.parquet": """
                SELECT paper_id, REPLACE(unnested_related, 'https://openalex.org/', '') AS related_id
                FROM (SELECT paper_id, unnest(related_works) AS unnested_related FROM processed_df)
                WHERE unnested_related IS NOT NULL
            """,
            "citation_counts_data.parquet": """
                SELECT paper_id, unnested_counts.year AS citation_year, unnested_counts.cited_by_count AS yearly_citations
                FROM (SELECT paper_id, unnest(counts_by_year) AS unnested_counts FROM processed_df)
                WHERE unnested_counts.year IS NOT NULL AND unnested_counts.cited_by_count IS NOT NULL
            """,
            "keyword_data.parquet": """
                SELECT paper_id, REPLACE(unnested_keywords.id, 'https://openalex.org/', '') AS keyword_id, unnested_keywords.score AS score
                FROM (SELECT paper_id, unnest(keywords) AS unnested_keywords FROM processed_df)
                WHERE keyword_id IS NOT NULL
            """,
            "work.parquet": """
                SELECT * EXCLUDE (mesh, authorships, topics, SDGs, awards, funders, primary_location, keywords, 
                abstract_inverted_index, counts_by_year, referenced_works, related_works),
                rebuild_abstract(abstract_inverted_index) AS abstract FROM processed_df
            """
        }

        for filename, sql in queries.items():
            file_path = os.path.join(output_dir, filename)
            try:
                # Mesh data has a conditional check in original logic
                if filename == "mesh_data.parquet":
                    mesh_exists = con.execute("SELECT count(*) FROM processed_df WHERE array_length(mesh) > 0").fetchone()[0]
                    if mesh_exists == 0: continue

                con.execute(f"COPY ({sql}) TO '{file_path}' (FORMAT PARQUET)")
            except Exception as e:
                error_msg = f"FAILED to create {filename} for {input_path}. Reason: {str(e)}"
                logger.error(error_msg)
                log_to_report(error_msg)

        logger.info(f"✅ Success: {input_path} ({time.time()-start_time:.2f}s)")
    except Exception as e:
        error_main = f"CRITICAL failure in pipeline for {input_path}: {str(e)}"
        logger.error(error_main)
        log_to_report(error_main)
    finally:
        con.close()

def main():
    input_base = r"D:\EmerG-NeT\Phase_1"
    output_base = r"D:\EmerG-NeT\Phase_2"
    
    # Initialize/Clear report file at start
    with open(REPORT_FILE, "w") as f:
        f.write("OpenAlex Processing Report\n==========================\n")

    if not os.path.exists(input_base):
        return

    folders = sorted([f.path for f in os.scandir(input_base) if "updated_date=" in f.name])[:200]
    for folder in folders:
        output_folder = os.path.join(output_base, os.path.basename(folder))
        files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
        for f in files:
            subfolder = os.path.join(output_folder, os.path.splitext(os.path.basename(f))[0])
            run_pipeline(f, subfolder)

if __name__ == "__main__":
    main()

2026-04-26 23:05:38,440 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-06-24\part_0000.parquet (0.20s)
2026-04-26 23:05:38,499 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-07-22\part_0000.parquet (0.05s)
2026-04-26 23:05:38,623 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-08-23\part_0000.parquet (0.12s)
2026-04-26 23:05:38,669 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-09-16\part_0000.parquet (0.04s)
2026-04-26 23:05:38,714 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-09-23\part_0000.parquet (0.04s)
2026-04-26 23:05:38,759 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-09-30\part_0000.parquet (0.04s)
2026-04-26 23:05:38,803 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-10-07\part_0000.parquet (0.04s)
2026-04-26 23:05:38,852 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-10-14\part_0000.parquet (0.05s)
2026-04-26 23:05:38,867 - WARNING - SKIP: File D:\EmerG-NeT\Phase_1\updated_date

# all language


In [13]:
import os
import logging
import time
import glob
import duckdb
import json
import traceback

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Path for the failure/skip report
REPORT_FILE = r"D:\EmerG-NeT\processing_report.txt"

def log_to_report(message):
    with open(REPORT_FILE, "a") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} - {message}\n")

def rebuild_abstract_duckdb(inverted_index):
    if not inverted_index:
        return ""
    try:
        data = inverted_index
        if isinstance(inverted_index, str):
            data = json.loads(inverted_index)
        if not data:
            return ""
        max_idx = 0
        for indices in data.values():
            if indices:
                local_max = max(indices)
                if local_max > max_idx:
                    max_idx = local_max
        if max_idx > 30000: 
            return ""
        result = [""] * (max_idx + 1)
        for word, indices in data.items():
            for idx in indices:
                if 0 <= idx < len(result):
                    result[idx] = str(word)
        return " ".join(result).strip()
    except Exception:
        return ""

def run_pipeline(input_path: str, output_dir: str):
    start_time = time.time()
    con = duckdb.connect(':memory:')
    con.create_function("rebuild_abstract", rebuild_abstract_duckdb, return_type='VARCHAR')

    col_to_keep = [
        'id', 'display_name', 'publication_date', 'language', 'type',
        'authorships', 'authors_count', 'topics', 'keywords',
        'locations_count', 'primary_location', 'sustainable_development_goals',
        'awards', 'funders', 'countries_distinct_count', 'institutions_distinct_count',
        'open_access', 'is_paratext', 'is_retracted',
        'referenced_works', 'referenced_works_count', 'related_works',
        'abstract_inverted_index', 'cited_by_count', 'counts_by_year', 'fwci',
        'citation_normalized_percentile', 'cited_by_percentile_year', 'mesh'
    ]
    
    cols_str = ", ".join([f'"{c}"' for c in col_to_keep if c != 'abstract_inverted_index'])

    try:
        # # --- ENGLISH LANGUAGE CHECK ---
        # # We check if there are ANY English records in this specific parquet file
        # has_english = con.execute(f"SELECT count(*) FROM read_parquet('{input_path}') WHERE language = 'en'").fetchone()[0]
        
        # if has_english == 0:
        #     msg = f"SKIP: File {input_path} does not contain English language records. Processing aborted."
        #     logger.warning(msg)
        #     log_to_report(msg)
        #     return

        con.execute(f"""
            CREATE VIEW raw_df AS 
            SELECT {cols_str}, abstract_inverted_index::JSON AS abstract_inverted_index
            FROM read_parquet('{input_path}')
        """)
        
        actual_date = None
        parts = os.path.normpath(input_path).split(os.sep)
        date_part = [p for p in parts if "updated_date=" in p]
        if date_part:
            actual_date = date_part[0].split("=")[1]
        date_sql = f"CAST('{actual_date}' AS DATE)" if actual_date else "NULL"

        con.execute(f"""
            CREATE TABLE clean_df AS
            SELECT 
                REPLACE(id, 'https://openalex.org/', '') AS paper_id,
                display_name AS paper_title,
                publication_date,
                type,
                authorships,
                authors_count::BIGINT AS authors_count,
                topics,
                keywords,
                language,
                locations_count::BIGINT AS locations_count,
                primary_location,
                sustainable_development_goals AS SDGs,
                awards,
                funders,
                countries_distinct_count::BIGINT AS countries_distinct_count,
                institutions_distinct_count::BIGINT AS institutions_distinct_count,
                open_access,
                referenced_works,
                referenced_works_count::BIGINT AS referenced_works_count,
                related_works,
                abstract_inverted_index,
                cited_by_count::BIGINT AS cited_by_count,
                counts_by_year,
                COALESCE(fwci, 1.0)::FLOAT AS fwci_val,
                citation_normalized_percentile,
                cited_by_percentile_year,
                mesh,
                {date_sql} AS updated_date
            FROM raw_df
            WHERE is_paratext = FALSE AND is_retracted = FALSE
        """)

        con.execute("""
            CREATE TABLE processed_df AS
            SELECT 
                * EXCLUDE (open_access, citation_normalized_percentile, cited_by_percentile_year),
                CAST(open_access.is_oa AS BOOLEAN) AS is_oa,
                open_access.oa_status AS oa_status,
                CASE 
                    WHEN cited_by_count = 0 THEN 0.0
                    ELSE COALESCE(CAST(citation_normalized_percentile.value AS FLOAT), 0.0)
                END AS percentile_rank,
                ((COALESCE(cited_by_percentile_year.min::FLOAT, 0.0) + COALESCE(cited_by_percentile_year.max::FLOAT, 0.0)) / 2.0)::FLOAT AS cited_by_percentile_year_avg
            FROM clean_df
        """)

        os.makedirs(output_dir, exist_ok=True)
        
        # --- SAFETY MEASURES FOR PARQUET CREATION ---
        
        queries = {
            "authors_data.parquet": """
                SELECT paper_id, REPLACE(a.author.id, 'https://openalex.org/', '') as author_id 
                FROM (SELECT paper_id, unnest(authorships) as a FROM processed_df)
            """,
            "topic_data.parquet": """
                SELECT paper_id, REPLACE(t.id, 'https://openalex.org/', '') as topic_id 
                FROM (SELECT paper_id, unnest(topics) as t FROM processed_df)
            """,
            "mesh_data.parquet": "SELECT paper_id, mesh FROM processed_df WHERE array_length(mesh) > 0",
            "awards_data.parquet": "SELECT paper_title, SDGs, awards, funders FROM processed_df",
            "prime_location_data.parquet": """
                SELECT paper_id, REPLACE(primary_location.source.id, 'https://openalex.org/', '') AS source_id,
                primary_location.source.display_name AS source_name,
                REPLACE(primary_location.source.host_organization, 'https://openalex.org/', '') AS host_org_id,
                primary_location.source.host_organization_name AS host_org_name
                FROM processed_df WHERE primary_location.source.id IS NOT NULL
            """,
            "cited_data.parquet": """
                SELECT paper_id, REPLACE(unnested_refs, 'https://openalex.org/', '') AS cited_id
                FROM (SELECT paper_id, unnest(referenced_works) AS unnested_refs FROM processed_df)
                WHERE unnested_refs IS NOT NULL
            """,
            "related_data.parquet": """
                SELECT paper_id, REPLACE(unnested_related, 'https://openalex.org/', '') AS related_id
                FROM (SELECT paper_id, unnest(related_works) AS unnested_related FROM processed_df)
                WHERE unnested_related IS NOT NULL
            """,
            "citation_counts_data.parquet": """
                SELECT paper_id, unnested_counts.year AS citation_year, unnested_counts.cited_by_count AS yearly_citations
                FROM (SELECT paper_id, unnest(counts_by_year) AS unnested_counts FROM processed_df)
                WHERE unnested_counts.year IS NOT NULL AND unnested_counts.cited_by_count IS NOT NULL
            """,
            "keyword_data.parquet": """
                SELECT paper_id, REPLACE(unnested_keywords.id, 'https://openalex.org/', '') AS keyword_id, unnested_keywords.score AS score
                FROM (SELECT paper_id, unnest(keywords) AS unnested_keywords FROM processed_df)
                WHERE keyword_id IS NOT NULL
            """,
            "work.parquet": """
                SELECT * EXCLUDE (mesh, authorships, topics, SDGs, awards, funders, primary_location, keywords, 
                abstract_inverted_index, counts_by_year, referenced_works, related_works),
                rebuild_abstract(abstract_inverted_index) AS abstract FROM processed_df
            """
        }

        for filename, sql in queries.items():
            file_path = os.path.join(output_dir, filename)
            try:
                # Mesh data has a conditional check in original logic
                if filename == "mesh_data.parquet":
                    mesh_exists = con.execute("SELECT count(*) FROM processed_df WHERE array_length(mesh) > 0").fetchone()[0]
                    if mesh_exists == 0: continue

                con.execute(f"COPY ({sql}) TO '{file_path}' (FORMAT PARQUET)")
            except Exception as e:
                error_msg = f"FAILED to create {filename} for {input_path}. Reason: {str(e)}"
                logger.error(error_msg)
                log_to_report(error_msg)

        logger.info(f"✅ Success: {input_path} ({time.time()-start_time:.2f}s)")

        try:
            os.remove(input_path)
            logger.info(f"Removed {input_path}")

        except Exception as e:
            msg=f"Failed to delete {input_path}"
            logger.error(msg)
            log_to_report(msg)

    except Exception as e:
        error_main = f"CRITICAL failure in pipeline for {input_path}: {str(e)}"
        logger.error(error_main)
        log_to_report(error_main)
    finally:
        con.close()

def main():
    input_base = r"D:\EmerG-NeT\Phase_1"
    output_base = r"D:\EmerG-NeT\Phase_2"
    
    # Initialize/Clear report file at start
    with open(REPORT_FILE, "w") as f:
        f.write("OpenAlex Processing Report\n==========================\n")

    if not os.path.exists(input_base):
        return

    folders = sorted([f.path for f in os.scandir(input_base) if "updated_date=" in f.name])
    for folder in folders:
        output_folder = os.path.join(output_base, os.path.basename(folder))
        files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
        for f in files:
            subfolder = os.path.join(output_folder, os.path.splitext(os.path.basename(f))[0])
            run_pipeline(f, subfolder)

if __name__ == "__main__":
    main()

2026-04-27 23:49:24,921 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-06-24\part_0000.parquet (0.22s)
2026-04-27 23:49:24,937 - INFO - Removed D:\EmerG-NeT\Phase_1\updated_date=2016-06-24\part_0000.parquet
2026-04-27 23:49:25,088 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-07-22\part_0000.parquet (0.15s)
2026-04-27 23:49:25,102 - INFO - Removed D:\EmerG-NeT\Phase_1\updated_date=2016-07-22\part_0000.parquet
2026-04-27 23:49:25,180 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-08-23\part_0000.parquet (0.07s)
2026-04-27 23:49:25,181 - INFO - Removed D:\EmerG-NeT\Phase_1\updated_date=2016-08-23\part_0000.parquet
2026-04-27 23:49:25,257 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-09-16\part_0000.parquet (0.07s)
2026-04-27 23:49:25,258 - INFO - Removed D:\EmerG-NeT\Phase_1\updated_date=2016-09-16\part_0000.parquet
2026-04-27 23:49:25,349 - INFO - ✅ Success: D:\EmerG-NeT\Phase_1\updated_date=2016-09-23\part_0000.parquet (0.09s)
2026-04-2

In [10]:
import pandas as pd

df = pd.read_parquet(r'D:\EmerG-NeT\Phase_2\updated_date=2016-06-24\part_0000\work.parquet')

df.head()

,paper_id,paper_title,publication_date,type,authors_count,language,locations_count,countries_distinct_count,institutions_distinct_count,referenced_works_count,cited_by_count,fwci_val,updated_date,is_oa,oa_status,percentile_rank,cited_by_percentile_year_avg,abstract
0,W146474571,ANALISIS PENGENDALIAN KUALITAS PADA PT. SEMARA...,2003-01-01,article,3,id,1,1,3,0,1,0.0,2016-06-24,True,green,0.032940,91.0,Mutu adalah salah satu masalah yang digambarka...
1,W127208130,Comparative Analysis of Net Realizable Value a...,1972-04-01,article,2,en,1,0,2,0,4,0.0,2016-06-24,False,closed,0.016785,91.5,Abstract This article presents information on ...
2,W286148134,PENGARUH MODEL PEMBELAJARAN INQUIRY TRAINING B...,2014-05-01,article,2,id,1,0,2,0,0,0.0,2016-06-24,True,green,0.000000,0.0,This study aimed to determine the effect of tr...
3,W210277063,A Study on the Feasibility of IGCC under the K...,2011-01-01,article,1,en,1,0,1,1,0,0.0,2016-06-24,False,closed,0.000000,0.0,An IGCC was evaluated as one of the next gener...
4,W65957438,Moral Reasoning: Should It Serve as a Criterio...,2001-01-01,article,1,en,2,0,1,38,4,0.0,2016-06-24,False,closed,0.022040,91.5,NaN


In [12]:
df.shape

(499, 18)

In [14]:
import os
import shutil
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

def copy_cited_data(source_base, target_base):
    """
    Traverses Phase_2 to find all cited_data.parquet files 
    and copies them to the target directory while maintaining structure.
    """
    if not os.path.exists(source_base):
        logger.error(f"Source directory {source_base} does not exist.")
        return

    # Counter for tracking progress
    copied_count = 0

    # Walk through the directory structure
    for root, dirs, files in os.walk(source_base):
        if "cited_data.parquet" in files:
            # Get the relative path from Phase_2 (e.g., 'updated_date=.../part_...')
            relative_path = os.path.relpath(root, source_base)
            
            # Construct the destination folder path
            destination_dir = os.path.join(target_base, relative_path)
            
            # Create the destination directory if it doesn't exist
            os.makedirs(destination_dir, exist_ok=True)
            
            # Define source and destination file paths
            src_file = os.path.join(root, "cited_data.parquet")
            dst_file = os.path.join(destination_dir, "cited_data.parquet")
            
            try:
                shutil.copy2(src_file, dst_file)
                copied_count += 1
                if copied_count % 50 == 0:
                    logger.info(f"Copied {copied_count} files so far...")
            except Exception as e:
                logger.error(f"Failed to copy {src_file}: {e}")

    logger.info(f"✅ Finished! Total files copied: {copied_count}")

if __name__ == "__main__":
    SOURCE = r"D:\EmerG-NeT\Phase_2"
    TARGET = r"D:\EmerG-NeT\cited_data"
    
    copy_cited_data(SOURCE, TARGET)


2026-04-29 16:12:21,150 - INFO - Copied 50 files so far...
2026-04-29 16:12:21,221 - INFO - Copied 100 files so far...
2026-04-29 16:12:21,264 - INFO - Copied 150 files so far...
2026-04-29 16:12:22,228 - INFO - Copied 200 files so far...
2026-04-29 16:12:24,376 - INFO - Copied 250 files so far...
2026-04-29 16:12:27,746 - INFO - Copied 300 files so far...
2026-04-29 16:12:30,774 - INFO - Copied 350 files so far...
2026-04-29 16:12:32,872 - INFO - Copied 400 files so far...
2026-04-29 16:12:48,297 - INFO - Copied 450 files so far...
2026-04-29 16:13:07,849 - INFO - Copied 500 files so far...
2026-04-29 16:13:29,366 - INFO - Copied 550 files so far...
2026-04-29 16:13:49,418 - INFO - Copied 600 files so far...
2026-04-29 16:14:09,224 - INFO - Copied 650 files so far...
2026-04-29 16:14:27,949 - INFO - Copied 700 files so far...
2026-04-29 16:14:45,217 - INFO - Copied 750 files so far...
2026-04-29 16:15:01,084 - INFO - Copied 800 files so far...
2026-04-29 16:15:20,798 - INFO - Copied 8

In [15]:
import os
import shutil
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

def move_and_rename_parquet(source_base, target_root):
    if not os.path.exists(source_base):
        logger.error(f"Source {source_base} not found.")
        return

    # This counter will increment every time we enter a new subfolder (e.g., part_0, part_1)
    # ensuring all files from the same source folder get the same index.
    folder_counter = 1

    # Walk through the Phase_2 structure
    # We sort to ensure consistency if you run this multiple times
    for root, dirs, files in os.walk(source_base):
        # We only care about folders that actually contain parquet files
        parquet_files = [f for f in files if f.endswith(".parquet")]
        
        if not parquet_files:
            continue
            
        logger.info(f"Processing folder index {folder_counter}: {root}")

        for filename in parquet_files:
            # 1. Determine the entity name (e.g., 'authors_data' from 'authors_data.parquet')
            entity_name = os.path.splitext(filename)[0]
            
            # 2. Create the specific destination folder (e.g., D:\EmerG-NeT\Data\authors_data)
            destination_folder = os.path.join(target_root, entity_name)
            os.makedirs(destination_folder, exist_ok=True)
            
            # 3. Construct the new filename (e.g., authors_data_1.parquet)
            # Or as per your request: author_1.parquet (removing the '_data' suffix if preferred)
            short_name = entity_name.replace("_data", "")
            new_filename = f"{short_name}_{folder_counter}.parquet"
            
            src_path = os.path.join(root, filename)
            dst_path = os.path.join(destination_folder, new_filename)
            
            try:
                # Use shutil.move to move the file. Use copy2 if you want to keep originals.
                shutil.move(src_path, dst_path)
            except Exception as e:
                logger.error(f"Error moving {filename}: {e}")

        # Increment the counter after processing all files in the current subfolder
        folder_counter += 1

    logger.info(f"✅ Reorganization complete. Total unique file sets processed: {folder_counter - 1}")

if __name__ == "__main__":
    SOURCE_DIR = r"D:\EmerG-NeT\Phase_2"
    TARGET_DIR = r"D:\EmerG-NeT\Data"
    
    move_and_rename_parquet(SOURCE_DIR, TARGET_DIR)

2026-04-29 16:28:13,782 - INFO - Processing folder index 1: D:\EmerG-NeT\Phase_2\updated_date=2016-06-24\part_0000
2026-04-29 16:28:13,789 - INFO - Processing folder index 2: D:\EmerG-NeT\Phase_2\updated_date=2016-07-22\part_0000
2026-04-29 16:28:13,794 - INFO - Processing folder index 3: D:\EmerG-NeT\Phase_2\updated_date=2016-08-23\part_0000
2026-04-29 16:28:13,797 - INFO - Processing folder index 4: D:\EmerG-NeT\Phase_2\updated_date=2016-09-16\part_0000
2026-04-29 16:28:13,801 - INFO - Processing folder index 5: D:\EmerG-NeT\Phase_2\updated_date=2016-09-23\part_0000
2026-04-29 16:28:13,805 - INFO - Processing folder index 6: D:\EmerG-NeT\Phase_2\updated_date=2016-09-30\part_0000
2026-04-29 16:28:13,809 - INFO - Processing folder index 7: D:\EmerG-NeT\Phase_2\updated_date=2016-10-07\part_0000
2026-04-29 16:28:13,815 - INFO - Processing folder index 8: D:\EmerG-NeT\Phase_2\updated_date=2016-10-14\part_0000
2026-04-29 16:28:13,819 - INFO - Processing folder index 9: D:\EmerG-NeT\Phase_2